# Задание 7. Создание анализатора грамматических родов

Признаки: последние N букв имени, N от 1 до 5. Классификатор: наивный байесовский (`nltk.NaiveBayesClassifier`). Данные: корпус `names` из NLTK (мужские и женские имена).

In [1]:
import random
import nltk
from nltk import NaiveBayesClassifier
from nltk.classify import accuracy as nltk_accuracy
from nltk.corpus import names

nltk.download("names", quiet=True)

[nltk_data] Error loading names: Security Violation [pathsec.urlopen]:
[nltk_data]     refusing a proxied fetch of
[nltk_data]     'https://raw.githubusercontent.com/nltk/nltk_data/gh-
[nltk_data]     pages/index.xml'. A configured proxy performs the
[nltk_data]     egress, so NLTK cannot pin the validated IP and SSRF
[nltk_data]     protection cannot be enforced (CWE-918). If and only
[nltk_data]     if the proxy is trusted to be SSRF-safe, opt in via
[nltk_data]     NLTK_ALLOW_PROXIED_URLOPEN=1 or
[nltk_data]     nltk.pathsec.ALLOW_PROXIED_FETCH=True.


False

In [2]:
# Функция, извлекающая последние N букв имени
def gender_features(word, N):
    return {"suffix": word[-N:].lower()}

In [3]:
male_list = [(name, "male") for name in names.words("male.txt")]
female_list = [(name, "female") for name in names.words("female.txt")]
data = male_list + female_list

random.seed(42)
random.shuffle(data)

print("Мужских имен:", len(male_list), "женских:", len(female_list))

Мужских имен: 2943 женских: 5001


In [4]:
# Ручная выборка для финального тестирования (имен нет в корпусе), с ожидаемым полом
test_names = {
    "Ksenia": "female",
    "Yasmin": "female",
    "Kenji": "male",
    "Bodhi": "male",
    "Oksana": "female",
    "Lyudmila": "female",
    "Gunnar": "male",
    "Dagny": "female",
}

allnames = set(names.words("male.txt")) | set(names.words("female.txt"))
print("Все тестовые имена отсутствуют в корпусе:", all(n not in allnames for n in test_names))

Все тестовые имена отсутствуют в корпусе: True


In [5]:
# Разбиение 80/20
n_train = int(0.8 * len(data))
train_raw, test_raw = data[:n_train], data[n_train:]
print("Обучающая выборка:", len(train_raw), "тестовая выборка:", len(test_raw))

results = {}
for N in range(1, 6):
    train_set = [(gender_features(n, N), g) for n, g in train_raw]
    test_set = [(gender_features(n, N), g) for n, g in test_raw]

    # Обучение наивного байесовского классификатора
    classifier = NaiveBayesClassifier.train(train_set)

    # Точность на тестовой выборке
    acc = nltk_accuracy(classifier, test_set)
    results[N] = acc
    print(f"N={N}: accuracy = {acc:.4f}")

    # Предсказание пола для каждого имени из ручной выборки
    for name, expected in test_names.items():
        pred = classifier.classify(gender_features(name, N))
        mark = "+" if pred == expected else "-"
        print(f"   {name:<10}{pred:<8}ожидаемо: {expected:<8}{mark}")
    print()

best_N = max(results, key=results.get)
print("Оптимальное N по точности на тестовой выборке:", best_N)

Обучающая выборка: 6355 тестовая выборка: 1589
N=1: accuracy = 0.7615
   Ksenia    female  ожидаемо: female  +
   Yasmin    male    ожидаемо: female  -
   Kenji     female  ожидаемо: male    -
   Bodhi     female  ожидаемо: male    -
   Oksana    female  ожидаемо: female  +
   Lyudmila  female  ожидаемо: female  +
   Gunnar    male    ожидаемо: male    +
   Dagny     female  ожидаемо: female  +

N=2: accuracy = 0.7867
   Ksenia    female  ожидаемо: female  +
   Yasmin    male    ожидаемо: female  -
   Kenji     female  ожидаемо: male    -
   Bodhi     female  ожидаемо: male    -
   Oksana    female  ожидаемо: female  +
   Lyudmila  female  ожидаемо: female  +
   Gunnar    male    ожидаемо: male    +
   Dagny     female  ожидаемо: female  +

N=3: accuracy = 0.7590
   Ksenia    female  ожидаемо: female  +
   Yasmin    female  ожидаемо: female  +
   Kenji     male    ожидаемо: male    +
   Bodhi     female  ожидаемо: male    -
   Oksana    female  ожидаемо: female  +
   Lyudmila  female  